# Clustering (et exploitation des données géographiques)

Dans cette partie nous exploitons les données géographiques et réalisons le clustering. 

Nous exportons des statistiques par commune pour croisement avec les données de vote. 

In [1]:
!pip install -r requirements.txt

In [2]:
from BDTopo_fonctions import load_gpkg
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg") #BASE ENREGISTREE AVEC SEULES ADRESSES PRECISEMENT LOCALISABLES (NUM VOIE = 1/3 SITADEL)

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (316142 lignes)


In [74]:
gdf.groupby("DEP_CODE").size().sort_values(ascending=False)

DEP_CODE
59    13700
62     8690
69     7830
67     6354
77     6317
      ...  
90      922
9       884
4       832
48      653
5       492
Length: 94, dtype: int64

In [ ]:
import warnings
from sklearn.exceptions import EfficiencyWarning
warnings.filterwarnings("ignore", category=EfficiencyWarning)
import importlib
import clustering_fonctions
importlib.reload(clustering_fonctions)
from clustering_fonctions import run_dbscan_parallele
temp=run_dbscan_parallele(gdf,500,4)

In [33]:
subset =temp[(temp["cluster_id_2025"]==-1) & (temp["Base"]=="Sitadel")].copy()

# Reprojection vers WGS84 (latitude / longitude)
subset = subset.to_crs(epsg=4326)

# Calcul du centroïde
subset["centroid"] = subset.geometry.centroid
subset["lat"] = subset.centroid.y
subset["lon"] = subset.centroid.x

subset[['lat','lon','SURF_CREEE','Annee_REF',"DEP_CODE"]][subset["SURF_CREEE"]>5000].sample(20)

/tmp/ipykernel_39216/1360183535.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["centroid"] = subset.geometry.centroid
/tmp/ipykernel_39216/1360183535.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lat"] = subset.centroid.y
/tmp/ipykernel_39216/1360183535.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lon"] = subset.centroid.x


,lat,lon,SURF_CREEE,Annee_REF,DEP_CODE
240995,45.098967,5.720076,10766.0,2020,38
150616,49.176340,6.879225,17428.0,2018,14
314567,51.007422,2.204660,42377.0,2020,59
21250,45.386253,1.569278,5855.0,2022,19
289419,48.728801,2.362307,10780.0,2021,94
211945,48.330640,-1.203895,7958.0,2016,35
41701,48.026925,3.915575,8500.0,2019,10
115019,48.095027,0.239158,6925.0,2024,72
68675,43.931270,4.777747,11930.0,2016,30
146331,49.206522,2.119973,49642.0,2017,60


In [35]:
temp.loc[150616]

Base                                                             Sitadel
DEP_CODE                                                              14
COMM                                                               14333
DATE_REELLE_AUTORISATION                                      11/05/2018
DATE_REELLE_DOC                                               01/03/2021
DATE_REELLE_DAACT                                                   None
SURF_CREEE                                                       17428.0
I_EXTENSION                                                        False
DESTINATION_PRINCIPALE                                                 4
ID                                                                  None
ORIGIN_BAT                                                          None
NATURE                                                              None
USAGE1                                                              None
SOURCE                                             

In [15]:
# Cluster_id présents dans Sitadel
sitadel_ids = temp.loc[temp["Base"]=="Sitadel", "cluster_id_2025"].unique()

# Comptage des occurrences de tous les bâtiments pour ces cluster_id
vc = temp["cluster_id_2025"].value_counts()

# Filtrer uniquement les clusters présents dans Sitadel
vc_sitadel = vc[vc.index.isin(sitadel_ids)]

# Nombre de clusters de taille 
vc_sitadel[vc_sitadel.index > 0]

cluster_id_2025
5900025    3910
9300001    3202
6900002    2384
9400001    2077
9200003    1130
           ... 
7300057       5
3800127       5
2500061       5
5900343       5
7300073       5
Name: count, Length: 3818, dtype: int64

In [16]:
# CLUSTER TOTAL, TOUS BAT EXISTANTS OU AUTORISES, ANNEE PAR ANNEE 
import pandas as pd
temp_sorted = temp.sort_values("Annee_REF")
results = []
for annee in sorted(temp_sorted["Annee_REF"].unique()):
    mask = temp_sorted["Annee_REF"] <= annee  # toutes les années <= annee
    cluster_col = temp_sorted.loc[mask, "cluster_id"]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    results.append({
        "Annee": annee,
        "pct_cluster_cumule": pct_cluster
    })
df_cumule = pd.DataFrame(results)
print(df_cumule)

    Annee  pct_cluster_cumule
0    2013           65.368863
1    2014           65.155076
2    2015           64.889324
3    2016           64.607099
4    2017           64.511624
5    2018           64.529133
6    2019           66.517363
7    2020           66.541428
8    2021           66.667758
9    2022           66.864188
10   2023           66.972262
11   2024           67.032456
12   2025           67.111614


In [17]:
#PC QUI EN 2025 SONT DANS UN CLUSTER
seuil=1000
temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)].groupby("Annee_REF")["cluster_id"].apply(lambda x: (x > 0).sum() / len(x) * 100)

Annee_REF
2014    82.734237
2015    82.803632
2016    83.772166
2017    84.111617
2018    86.802326
2019    85.310387
2020    85.356304
2021    87.941372
2022    87.836991
2023    86.732373
2024    83.247423
2025    85.590062
Name: cluster_id, dtype: float64

In [18]:
#PC QUI SONT DANS UN CLUSTER AU MOMENT DE AUTORISATION 

temp_sitadel = temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)]

for annee in sorted(temp_sitadel["Annee_REF"].unique()):
    col = f"cluster_id_{annee}"
    mask = temp_sitadel["Annee_REF"] == annee
    cluster_col = temp_sitadel.loc[mask, col]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    print(f"Année {annee}: {pct_cluster:.1f}% des bâtiments clusterisés")

Année 2014: 76.8% des bâtiments clusterisés
Année 2015: 77.3% des bâtiments clusterisés
Année 2016: 79.7% des bâtiments clusterisés
Année 2017: 80.3% des bâtiments clusterisés
Année 2018: 81.9% des bâtiments clusterisés
Année 2019: 82.7% des bâtiments clusterisés
Année 2020: 83.7% des bâtiments clusterisés
Année 2021: 86.1% des bâtiments clusterisés
Année 2022: 87.0% des bâtiments clusterisés
Année 2023: 85.9% des bâtiments clusterisés
Année 2024: 82.9% des bâtiments clusterisés
Année 2025: 85.6% des bâtiments clusterisés


In [19]:
annees = sorted(temp["Annee_REF"].unique())
result_list = []
for annee in annees:
    col = f"cluster_id_{annee}"
    if col in temp.columns:
        # compter les clusters uniques non-négatifs
        nb_clusters = temp[col][temp[col] > 0].nunique()
        result_list.append({"Annee_REF": annee, "nb_clusters": nb_clusters})

df_clusters_par_annee = pd.DataFrame(result_list)
print(df_clusters_par_annee)

    Annee_REF  nb_clusters
0        2013         6351
1        2014         6570
2        2015         6757
3        2016         6912
4        2017         7081
5        2018         7169
6        2019         8455
7        2020         8621
8        2021         8684
9        2022         8783
10       2023         8839
11       2024         8917
12       2025         8991
